# C4-Modeler-LLM-Agents: Interactive Experiment Runner

This notebook provides an interactive interface for running and analyzing C4 modeling experiments with enhanced evaluation frameworks for journal publication research.

## Overview

This notebook implements:
- **Environment Setup**: Secure API key management with dotenv
- **LangSmith Tracing**: Full observability for LLM interactions
- **Experiment Execution**: Five experimental conditions (role ablation, interaction order, specialization, shared memory, diversity)
- **Results Analysis**: Statistical analysis and visualization capabilities
- **Enhanced Evaluation**: ISO/IEC 25010, ATAM scenarios, and architectural soundness metrics

## Research Context

This system extends the HICSS 2025 paper on multi-agent C4 modeling with enhanced experimental frameworks for journal publication, focusing on:
1. Role ablation studies
2. Interaction order dynamics
3. Agent specialization impacts
4. Shared memory effectiveness  
5. Team diversity analysis

# 1. Environment Setup and Configuration

First, we'll configure the environment with proper API keys and settings.

In [1]:
# Environment Setup with Enhanced Security
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path for imports
parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir / "src"))

print("🔧 Setting up environment...")
print(f"📁 Working directory: {Path.cwd()}")
print(f"📂 Parent directory: {parent_dir}")

# Verify critical files exist
config_file = parent_dir / "experiment_configs.yaml"
env_file = parent_dir / ".env"

if not config_file.exists():
    print("❌ experiment_configs.yaml not found!")
else:
    print("✅ experiment_configs.yaml found")

if not env_file.exists():
    print("❌ .env file not found!")
else:
    print("✅ .env file found")

🔧 Setting up environment...
📁 Working directory: c:\Users\kamil\Documents\Kamil\HICSS extension\repo\C4-Modeler-LLM-Agents\notebooks
📂 Parent directory: c:\Users\kamil\Documents\Kamil\HICSS extension\repo\C4-Modeler-LLM-Agents
✅ experiment_configs.yaml found
✅ .env file found


In [2]:
# Load Environment Variables with Security Best Practices
from dotenv import load_dotenv
import os

# Change to parent directory for .env file loading
os.chdir(parent_dir)
load_dotenv()

# Configure API Keys with validation
print("🔑 Configuring API Keys...")

api_keys = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "DEEPSEEK_API_KEY": os.getenv("DEEPSEEK_API_KEY"), 
    "GOOGLE_API_KEY": os.getenv("GOOGLE_API_KEY"),
    "XAI_API_KEY": os.getenv("XAI_API_KEY")
}

# Validate and set API keys
for key_name, key_value in api_keys.items():
    if key_value:
        os.environ[key_name] = key_value
        print(f"✅ {key_name}: {'*' * (len(key_value) - 10) + key_value[-10:]}")
    else:
        print(f"⚠️ {key_name}: Not set")

# Configure LangChain/LangSmith Tracing
print("\n🔍 Configuring LangSmith Tracing...")

# Set tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "C4 Modeler Agents - Interactive")

# Validate tracing setup
if os.environ.get("LANGCHAIN_API_KEY"):
    print(f"✅ LangSmith Tracing: Enabled")
    print(f"📊 Project: {os.environ.get('LANGCHAIN_PROJECT')}")
    print(f"🔗 Tracing V2: {os.environ.get('LANGCHAIN_TRACING_V2')}")
else:
    print("⚠️ LangSmith API Key not found - tracing will be disabled")

print("\n🎯 Environment setup complete!")

🔑 Configuring API Keys...
✅ OPENAI_API_KEY: **********************************************JEpX23SuUG
✅ DEEPSEEK_API_KEY: *************************da13d477eb
✅ GOOGLE_API_KEY: *****************************TuiQDKNeWo
✅ XAI_API_KEY: **************************************************************************mniTKHlBVr

🔍 Configuring LangSmith Tracing...
✅ LangSmith Tracing: Enabled
📊 Project: C4 Modeler Agents - Extended
🔗 Tracing V2: true

🎯 Environment setup complete!


# 2. LangSmith Tracing Initialization

Configure LangSmith tracing for comprehensive observability of our experiments.

In [3]:
# Initialize LangSmith Tracing with Advanced Configuration
import langsmith as ls
from datetime import datetime
import uuid

# Create a unique session ID for this notebook run
session_id = str(uuid.uuid4())[:8]
session_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("🔍 Initializing LangSmith Tracing...")

# Configure tracing context for this experiment session
def setup_experiment_tracing(experiment_name="interactive", model_name="gpt-4o-mini"):
    """Set up LangSmith tracing context for experiments."""
    
    project_name = f"C4-Experiments-{experiment_name}-{session_id}"
    
    # Create tracing context with rich metadata
    return ls.tracing_context(
        project_name=project_name,
        enabled=True,
        tags=[
            experiment_name, 
            model_name, 
            "jupyter-notebook",
            "interactive-session",
            f"session-{session_id}"
        ],
        metadata={
            "session_id": session_id,
            "session_timestamp": session_timestamp,
            "experiment_type": experiment_name,
            "model": model_name,
            "environment": "jupyter-notebook",
            "purpose": "journal-extension-research"
        }
    )

# Test tracing connection
try:
    if os.environ.get("LANGCHAIN_API_KEY"):
        # Create a test client to verify connection
        client = ls.Client()
        print("✅ LangSmith connection verified")
        print(f"📊 Session ID: {session_id}")
        print(f"⏰ Session started: {session_timestamp}")
    else:
        print("⚠️ LangSmith tracing disabled (no API key)")
        
except Exception as e:
    print(f"❌ LangSmith connection failed: {str(e)}")
    print("🔄 Experiments will run without tracing")

# Helper function to wait for traces to finish
def wait_for_traces():
    """Ensure all traces are submitted before proceeding."""
    try:
        from langchain_core.tracers.langchain import wait_for_all_tracers
        wait_for_all_tracers()
        print("✅ All traces submitted")
    except ImportError:
        print("ℹ️ Trace waiting not available")

print("🎯 Tracing initialization complete!")

🔍 Initializing LangSmith Tracing...
✅ LangSmith connection verified
📊 Session ID: 7e9658d7
⏰ Session started: 20251022_140744
🎯 Tracing initialization complete!


# 3. Experiment Selection and Configuration

Load experiment configurations and provide interactive selection interface.

In [4]:
from c4modeler.experiments import EnhancedExperimentRunner
from c4modeler.integration import validate_experiment_condition
import yaml
import pandas as pd
from IPython.display import display, HTML

# Initialize experiment runner
print("🧪 Initializing Enhanced Experiment Runner...")

try:
    runner = EnhancedExperimentRunner("experiment_configs.yaml")
    print("✅ Experiment runner initialized successfully")
    
    # Load system briefs
    runner.load_system_briefs("data/briefs")
    print(f"📚 Loaded {len(runner.briefs)} system briefs: {list(runner.briefs.keys())}")
    
except Exception as e:
    print(f"❌ Failed to initialize experiment runner: {str(e)}")
    raise

# Validate configuration
if not runner.config:
    print("❌ No experiment configuration loaded!")
    raise ValueError("Failed to load experiment configuration")

print(f"✅ Loaded {len(runner.config.get('experiments', {}))} experiments")

🧪 Initializing Enhanced Experiment Runner...
✅ Experiment runner initialized successfully
📚 Loaded 5 system briefs: ['clinic-management-system', 'library-management-system', 'nextgen-pos', 'online-bookstore', 'student-information-system']
📚 Loaded 5 system briefs: ['clinic-management-system', 'library-management-system', 'nextgen-pos', 'online-bookstore', 'student-information-system']
✅ Loaded 5 experiments


In [5]:
# Display Available Experiments
def display_experiments_table():
    """Create an interactive table of available experiments."""
    
    experiments = []
    for exp_name, exp_config in runner.config.get("experiments", {}).items():
        experiments.append({
            "Experiment ID": exp_name,
            "Name": exp_config["name"],
            "Description": exp_config["description"][:80] + "..." if len(exp_config["description"]) > 80 else exp_config["description"],
            "Conditions": len(exp_config.get("conditions", [])),
            "Purpose": exp_config["purpose"][:60] + "..." if len(exp_config["purpose"]) > 60 else exp_config["purpose"],
            "Hypothesis": exp_config.get("hypothesis", "")[:80] + "..." if len(exp_config.get("hypothesis", "")) > 80 else exp_config.get("hypothesis", "")
        })
    
    df = pd.DataFrame(experiments)
    return df

# Display experiments
experiments_df = display_experiments_table()
print("📋 Available Experiments:")
print("=" * 100)
display(experiments_df)

# Configuration options
print("\n⚙️ Configuration Options:")
print("Recommended Model: gpt-4o-mini (reliable and cost-effective)")
print("Available Models: gpt-4o-mini, gpt-4o, gemini-1.5-flash-latest, gemini-2.5-flash")
print("Available Datasets:", list(runner.briefs.keys()))
print("Rounds per condition:", runner.config.get("protocol", {}).get("rounds_per_condition", 3))
print("\n💡 Cost Estimate with GPT-4o-mini: ~$30-50 for all 300 experimental runs")

📋 Available Experiments:


,Experiment ID,Name,Description,Conditions,Purpose,Hypothesis
0,role_ablation,Role Ablation Study,Remove one expert role and evaluate impact on ...,5,Determine which roles are most critical for ar...,Removing domain-oriented roles (PO) reduces fu...
1,interaction_order,Role Interaction Order Study,Vary the speaking order to study influence of ...,4,Understand how conversation sequencing affects...,Architect-first yields higher structural valid...
2,role_specialization,Role Specialization vs. Generic Study,Compare specialized personas against generic s...,3,Test whether role specialization adds meaningf...,Specialized teams outperform generic ones on f...
3,shared_memory,Shared Memory and Glossary Study,Evaluate impact of persistent shared knowledge...,4,Test whether shared glossary improves architec...,Glossary-based collaboration significantly imp...
4,role_diversity,Role Diversity and Cross-Level Consistency Study,Examine how team heterogeneity affects C4 leve...,4,Quantify how diversity of expertise contribute...,Diverse teams yield higher abstraction coheren...



⚙️ Configuration Options:
Recommended Model: gpt-4o-mini (reliable and cost-effective)
Available Models: gpt-4o-mini, gpt-4o, gemini-1.5-flash-latest, gemini-2.5-flash
Available Datasets: ['clinic-management-system', 'library-management-system', 'nextgen-pos', 'online-bookstore', 'student-information-system']
Rounds per condition: 3

💡 Cost Estimate with GPT-4o-mini: ~$30-50 for all 300 experimental runs


# 4. Single Experiment Runner

Execute individual experimental conditions with detailed monitoring.

In [6]:
# Single Experiment Execution with Live Monitoring
import json
from datetime import datetime
import time

def run_single_experiment_with_tracing(experiment_name, model_name="gpt-4o-mini", live_updates=True):
    """Run a single experiment with comprehensive tracing and monitoring."""
    
    print(f"🚀 Starting Experiment: {experiment_name}")
    print(f"🤖 Model: {model_name}")
    print(f"⏰ Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    
    # Set up experiment-specific tracing
    with setup_experiment_tracing(experiment_name, model_name):
        try:
            start_time = time.time()
            
            # Run the experiment
            results = runner.run_experiment(experiment_name, model_name)
            
            execution_time = time.time() - start_time
            
            # Display summary
            print(f"\n✅ Experiment completed successfully!")
            print(f"⏱️ Execution time: {execution_time:.1f} seconds")
            
            if "conditions" in results:
                conditions = results["conditions"]
                print(f"📊 Summary:")
                print(f"   Total Conditions: {len(conditions)}")
                
                for condition_id, condition_data in conditions.items():
                    summary = condition_data.get("condition_summary", {})
                    runs = summary.get("total_runs", 0)
                    print(f"   {condition_id}: {runs} runs")
            
            # Save results with timestamp
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            results_file = f"data/results/notebook_experiments/{experiment_name}_{timestamp}.json"
            
            # Ensure directory exists
            Path(results_file).parent.mkdir(parents=True, exist_ok=True)
            
            with open(results_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            
            print(f"💾 Results saved to: {results_file}")
            
            # Wait for traces to complete
            wait_for_traces()
            
            return results
            
        except Exception as e:
            print(f"❌ Experiment failed: {str(e)}")
            raise

# Example: Uncomment and modify to run a specific experiment
# results = run_single_experiment_with_tracing("role_ablation", "gpt-4o-mini")

print("💡 To run an experiment, uncomment and execute:")
print("results = run_single_experiment_with_tracing('role_ablation', 'gpt-4o-mini')")

💡 To run an experiment, uncomment and execute:
results = run_single_experiment_with_tracing('role_ablation', 'gpt-4o-mini')


# 5. Batch Experiment Execution

Run multiple experiments or complete experimental suites.

In [7]:
# Batch Experiment Execution with Progress Tracking
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import threading

def run_batch_experiments(experiment_list, model_name="gpt-4o-mini", parallel=False, max_workers=2):
    """Run multiple experiments with progress tracking and error handling."""
    
    print(f"🚀 Starting Batch Execution")
    print(f"📋 Experiments: {experiment_list}")
    print(f"🤖 Model: {model_name}")
    print(f"🔄 Parallel: {parallel} ({'Enabled' if parallel else 'Sequential'})")
    print("=" * 80)
    
    all_results = {}
    failed_experiments = []
    
    # Set up batch tracing context
    with setup_experiment_tracing("batch-experiments", model_name):
        if parallel and len(experiment_list) > 1:
            # Parallel execution
            print(f"🏃‍♂️ Running {len(experiment_list)} experiments in parallel (max {max_workers} workers)")
            
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                # Submit all experiments
                future_to_experiment = {
                    executor.submit(runner.run_experiment, exp_name, model_name): exp_name
                    for exp_name in experiment_list
                }
                
                # Process results as they complete
                for future in tqdm(as_completed(future_to_experiment), total=len(experiment_list), desc="Experiments"):
                    exp_name = future_to_experiment[future]
                    try:
                        results = future.result()
                        all_results[exp_name] = results
                        print(f"✅ {exp_name} completed")
                    except Exception as e:
                        print(f"❌ {exp_name} failed: {str(e)}")
                        failed_experiments.append((exp_name, str(e)))
        else:
            # Sequential execution
            print(f"🚶‍♂️ Running {len(experiment_list)} experiments sequentially")
            
            for exp_name in tqdm(experiment_list, desc="Experiments"):
                try:
                    print(f"\n🧪 Running {exp_name}...")
                    results = runner.run_experiment(exp_name, model_name)
                    all_results[exp_name] = results
                    print(f"✅ {exp_name} completed")
                except Exception as e:
                    print(f"❌ {exp_name} failed: {str(e)}")
                    failed_experiments.append((exp_name, str(e)))
    
    # Summary
    print(f"\n📊 Batch Execution Summary:")
    print(f"   Successful: {len(all_results)}/{len(experiment_list)}")
    print(f"   Failed: {len(failed_experiments)}")
    
    if failed_experiments:
        print(f"\n❌ Failed Experiments:")
        for exp_name, error in failed_experiments:
            print(f"   {exp_name}: {error}")
    
    # Save batch results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    batch_results = {
        "metadata": {
            "timestamp": timestamp,
            "model_name": model_name,
            "parallel_execution": parallel,
            "total_experiments": len(experiment_list),
            "successful_experiments": len(all_results),
            "failed_experiments": len(failed_experiments)
        },
        "results": all_results,
        "failures": failed_experiments
    }
    
    results_file = f"data/results/notebook_experiments/batch_{timestamp}.json"
    Path(results_file).parent.mkdir(parents=True, exist_ok=True)
    
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(batch_results, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Batch results saved to: {results_file}")
    
    # Wait for all traces
    wait_for_traces()
    
    return batch_results

# Example usage
print("💡 Example batch execution:")
print("# Run specific experiments")
print("batch_results = run_batch_experiments(['role_ablation', 'interaction_order'], 'gpt-4o-mini')")
print("\n# Run all experiments")
print("all_experiments = list(runner.config.get('experiments', {}).keys())")
print("batch_results = run_batch_experiments(all_experiments, 'gpt-4o-mini', parallel=True)")

💡 Example batch execution:
# Run specific experiments
batch_results = run_batch_experiments(['role_ablation', 'interaction_order'], 'gpt-4o-mini')

# Run all experiments
all_experiments = list(runner.config.get('experiments', {}).keys())
batch_results = run_batch_experiments(all_experiments, 'gpt-4o-mini', parallel=True)


# 6. Results Analysis and Visualization

Analyze experiment results with statistical summaries and publication-ready charts.

In [8]:
# Results Analysis and Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import glob

# Set up plotting style for publication quality
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

def load_experiment_results(results_dir="data/results/notebook_experiments"):
    """Load all experiment results from the results directory."""
    
    results_files = glob.glob(f"{results_dir}/*.json")
    all_results = {}
    
    print(f"📂 Loading results from: {results_dir}")
    print(f"🔍 Found {len(results_files)} result files")
    
    for file_path in results_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            filename = Path(file_path).stem
            
            # Handle both single experiment and batch results
            if "results" in data:  # Batch results
                for exp_name, exp_data in data["results"].items():
                    all_results[f"{exp_name}_{filename}"] = exp_data
            else:  # Single experiment result
                all_results[filename] = data
                
        except Exception as e:
            print(f"⚠️ Failed to load {file_path}: {str(e)}")
    
    print(f"✅ Loaded {len(all_results)} experiment results")
    return all_results

def extract_metrics_dataframe(results_dict):
    """Extract metrics into a pandas DataFrame for analysis."""
    
    metrics_data = []
    
    for result_name, result_data in results_dict.items():
        if "conditions" not in result_data:
            continue
            
        experiment_name = result_data.get("experiment_metadata", {}).get("name", result_name.split("_")[0])
        
        for condition_id, condition_data in result_data["conditions"].items():
            condition_summary = condition_data.get("condition_summary", {})
            averages = condition_summary.get("averages", {})
            metadata = condition_data.get("condition_metadata", {})
            
            # Extract key metrics
            metrics_row = {
                "experiment": experiment_name,
                "condition": condition_id,
                "condition_name": metadata.get("name", condition_id),
                "total_runs": condition_summary.get("total_runs", 0),
                
                # ISO 25010 metrics
                "iso_functional": averages.get("iso_functional_suitability", 0),
                "iso_security": averages.get("iso_security", 0),
                "iso_maintainability": averages.get("iso_maintainability", 0),
                "iso_reliability": averages.get("iso_reliability", 0),
                "iso_usability": averages.get("iso_usability", 0),
                
                # ATAM metrics
                "atam_responsiveness": averages.get("atam_responsiveness", 0),
                
                # Architectural soundness
                "coupling_density": averages.get("soundness_coupling_density", 0),
                "abstraction_coherence": averages.get("soundness_abstraction_coherence", 0),
                "semantic_completeness": averages.get("soundness_semantic_completeness", 0),
                
                # Metadata
                "roles": metadata.get("roles", []),
                "memory_type": metadata.get("memory_type", "none"),
                "speaking_order": metadata.get("speaking_order", [])
            }
            
            metrics_data.append(metrics_row)
    
    df = pd.DataFrame(metrics_data)
    return df

# Example: Load and analyze results
print("💡 To analyze results, run:")
print("results = load_experiment_results()")
print("metrics_df = extract_metrics_dataframe(results)")
print("display(metrics_df.head())")

💡 To analyze results, run:
results = load_experiment_results()
metrics_df = extract_metrics_dataframe(results)
display(metrics_df.head())


In [9]:
# Visualization Functions for Publication-Ready Charts
def create_experiment_comparison_chart(metrics_df, metric_col="iso_functional", experiment="role_ablation"):
    """Create publication-ready comparison charts."""
    
    # Filter for specific experiment
    exp_data = metrics_df[metrics_df["experiment"].str.contains(experiment, case=False, na=False)]
    
    if exp_data.empty:
        print(f"⚠️ No data found for experiment: {experiment}")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'Experiment Analysis: {experiment.replace("_", " ").title()}', fontsize=16, fontweight='bold')
    
    # 1. Bar chart of main metric by condition
    ax1 = axes[0, 0]
    exp_data.plot(x="condition", y=metric_col, kind="bar", ax=ax1, color="skyblue")
    ax1.set_title(f'{metric_col.replace("_", " ").title()} by Condition')
    ax1.set_ylabel('Score')
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Heatmap of all ISO metrics
    ax2 = axes[0, 1]
    iso_metrics = [col for col in exp_data.columns if col.startswith("iso_")]
    if iso_metrics:
        heatmap_data = exp_data.set_index("condition")[iso_metrics].T
        sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap="YlOrRd", ax=ax2)
        ax2.set_title('ISO 25010 Metrics Heatmap')
    
    # 3. Soundness metrics comparison
    ax3 = axes[1, 0]
    soundness_metrics = ["coupling_density", "abstraction_coherence", "semantic_completeness"]
    soundness_data = exp_data.set_index("condition")[soundness_metrics]
    soundness_data.plot(kind="bar", ax=ax3)
    ax3.set_title('Architectural Soundness Metrics')
    ax3.set_ylabel('Score')
    ax3.tick_params(axis='x', rotation=45)
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 4. Overall quality score (composite)
    ax4 = axes[1, 1]
    # Calculate composite score
    quality_cols = ["iso_functional", "iso_maintainability", "abstraction_coherence"]
    available_cols = [col for col in quality_cols if col in exp_data.columns]
    if available_cols:
        exp_data["composite_score"] = exp_data[available_cols].mean(axis=1)
        exp_data.plot(x="condition", y="composite_score", kind="bar", ax=ax4, color="lightgreen")
        ax4.set_title('Composite Quality Score')
        ax4.set_ylabel('Average Score')
        ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical summary
    print(f"\n📊 Statistical Summary for {experiment}:")
    summary_stats = exp_data[available_cols + [metric_col]].describe()
    display(summary_stats)

def export_results_to_csv(metrics_df, filename="experiment_results_export.csv"):
    """Export results to CSV for external analysis."""
    
    export_path = f"data/results/{filename}"
    Path(export_path).parent.mkdir(parents=True, exist_ok=True)
    
    metrics_df.to_csv(export_path, index=False)
    print(f"📤 Results exported to: {export_path}")
    
    return export_path

def generate_anova_analysis(metrics_df, metric="iso_functional", experiment="role_ablation"):
    """Perform ANOVA analysis for statistical significance."""
    
    exp_data = metrics_df[metrics_df["experiment"].str.contains(experiment, case=False, na=False)]
    
    if exp_data.empty or len(exp_data["condition"].unique()) < 2:
        print("⚠️ Insufficient data for ANOVA analysis")
        return
    
    # Group data by condition
    groups = [group[metric].values for name, group in exp_data.groupby("condition")]
    
    # Perform ANOVA
    f_stat, p_value = stats.f_oneway(*groups)
    
    print(f"📈 ANOVA Analysis for {metric} in {experiment}:")
    print(f"   F-statistic: {f_stat:.4f}")
    print(f"   P-value: {p_value:.4f}")
    print(f"   Significant: {'Yes' if p_value < 0.05 else 'No'} (α = 0.05)")
    
    return f_stat, p_value

# Interactive Analysis Example
print("💡 To run analysis:")
print("# Load results")
print("results = load_experiment_results()")
print("metrics_df = extract_metrics_dataframe(results)")
print("")
print("# Create visualizations")
print("create_experiment_comparison_chart(metrics_df, 'iso_functional', 'role_ablation')")
print("")
print("# Statistical analysis")
print("generate_anova_analysis(metrics_df, 'iso_functional', 'role_ablation')")
print("")
print("# Export results")
print("export_results_to_csv(metrics_df)")

💡 To run analysis:
# Load results
results = load_experiment_results()
metrics_df = extract_metrics_dataframe(results)

# Create visualizations
create_experiment_comparison_chart(metrics_df, 'iso_functional', 'role_ablation')

# Statistical analysis
generate_anova_analysis(metrics_df, 'iso_functional', 'role_ablation')

# Export results
export_results_to_csv(metrics_df)


# Quick Start Guide

## Running Your First Experiment

1. **Environment Check**: Ensure all cells above have been executed successfully
2. **Select Experiment**: Choose from the available experiments shown in Section 3
3. **Execute**: Use the single experiment runner or batch executor
4. **Analyze**: Use the visualization functions to analyze results

## Example Workflow with Gemini 2.5 Flash

```python
# 1. Run a quick test experiment with latest Gemini model
results = run_single_experiment_with_tracing("role_ablation", "gemini-2.5-flash-preview-05-20")

# 2. Load and analyze results
results_data = load_experiment_results()
metrics_df = extract_metrics_dataframe(results_data)

# 3. Create visualizations
create_experiment_comparison_chart(metrics_df, "iso_functional", "role_ablation")

# 4. Export for further analysis
export_results_to_csv(metrics_df)
```

## Model Selection: Gemini 2.5 Flash

- **Model**: `gemini-2.5-flash`
- **Advantages**: Latest Google model, excellent performance/cost ratio
- **Cost**: ~$50-80 for all 300 experimental runs with $300 credit
- **Speed**: Optimized for fast inference, perfect for research experiments

## Tips

- **LangSmith Tracing**: Check the LangSmith dashboard for detailed traces
- **Results Storage**: All results are automatically saved with timestamps
- **Error Handling**: Failed experiments are logged but won't stop batch execution
- **Resource Management**: Use parallel execution carefully to avoid API rate limits
- **Budget**: Your $300 Google credit will easily cover all experiments with plenty to spare

---

**Ready to start your experiments with Gemini 2.5 Flash!** 🚀

# 🎯 Complete Experimental Study: Journal Extension Research

This section provides a structured approach to execute all 300 experimental runs for journal publication research.

## Experimental Design Overview

- **Total Experiments**: 5 (role ablation, interaction order, specialization, shared memory, diversity)
- **Datasets**: 5 (clinic, library, pos, bookstore, student) 
- **Rounds per Condition**: 3 (for statistical significance)
- **Total Runs**: 300 experimental runs
- **Model**: `gemini-2.5-flash` (cost-effective with excellent performance)
- **Estimated Duration**: ~6-8 hours for complete study
- **Estimated Cost**: ~$50-80 with Google API credits

## Run Breakdown by Experiment

| Experiment | Conditions | Datasets | Rounds | Total Runs |
|------------|------------|----------|--------|------------|
| Role Ablation | 5 | 5 | 3 | **75** |
| Interaction Order | 4 | 5 | 3 | **60** |
| Specialization | 4 | 5 | 3 | **60** |
| Shared Memory | 3 | 5 | 3 | **45** |
| Diversity | 4 | 5 | 3 | **60** |
| **TOTAL** | **20** | **5** | **3** | **300** |

## 📋 Step 1: Pre-Experiment Setup and Validation

Before starting the complete experimental study, let's validate our environment and configuration.

In [10]:
# ========================================================================
# STEP 1: PRE-EXPERIMENT SETUP AND VALIDATION
# ========================================================================

import json
import time
from datetime import datetime, timedelta
import pandas as pd
from pathlib import Path

# Experimental configuration - UPDATED TO GEMINI-2.5-FLASH-LITE
EXPERIMENT_MODEL = "gemini-2.5-flash-lite"
EXPERIMENT_START_TIME = datetime.now()

print("🎯 COMPLETE EXPERIMENTAL STUDY SETUP")
print("=" * 80)
print(f"📅 Study Start Time: {EXPERIMENT_START_TIME.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🤖 Model: {EXPERIMENT_MODEL}")
print(f"📊 Total Planned Runs: 300")

# Validate environment setup
print("\n🔍 Environment Validation:")

# Check API keys
api_keys_status = {
    "OPENAI_API_KEY": "✅ Set" if os.environ.get("OPENAI_API_KEY") else "❌ Missing",
    "LANGCHAIN_API_KEY": "✅ Set" if os.environ.get("LANGCHAIN_API_KEY") else "⚠️ Missing (tracing disabled)"
}

for key, status in api_keys_status.items():
    print(f"   {key}: {status}")

# Validate experiment runner
if 'runner' not in locals():
    print("❌ Experiment runner not initialized. Please run Section 3 first.")
else:
    print("✅ Experiment runner initialized")
    
    # Check experiment configurations
    available_experiments = list(runner.config.get("experiments", {}).keys())
    available_briefs = list(runner.briefs.keys())
    
    print(f"✅ Available experiments: {len(available_experiments)} - {available_experiments}")
    print(f"✅ Available datasets: {len(available_briefs)} - {available_briefs}")
    
    # Calculate expected runs
    expected_runs = 0
    for exp_name in available_experiments:
        exp_config = runner.config["experiments"][exp_name]
        conditions = len(exp_config.get("conditions", []))
        datasets = len(available_briefs)
        rounds = runner.config.get("protocol", {}).get("rounds_per_condition", 3)
        exp_runs = conditions * datasets * rounds
        expected_runs += exp_runs
        print(f"   {exp_name}: {conditions} conditions × {datasets} datasets × {rounds} rounds = {exp_runs} runs")
    
    print(f"\n📊 Total Expected Runs: {expected_runs}")

# Create results directory
results_dir = Path("data/results/complete_study")
results_dir.mkdir(parents=True, exist_ok=True)
print(f"📁 Results directory: {results_dir.absolute()}")

print("\n🎯 Pre-experiment validation complete!")
print("=" * 80)

🎯 COMPLETE EXPERIMENTAL STUDY SETUP
📅 Study Start Time: 2025-10-22 14:07:47
🤖 Model: gemini-2.5-flash-lite
📊 Total Planned Runs: 300

🔍 Environment Validation:
   OPENAI_API_KEY: ✅ Set
   LANGCHAIN_API_KEY: ✅ Set
✅ Experiment runner initialized
✅ Available experiments: 5 - ['role_ablation', 'interaction_order', 'role_specialization', 'shared_memory', 'role_diversity']
✅ Available datasets: 5 - ['clinic-management-system', 'library-management-system', 'nextgen-pos', 'online-bookstore', 'student-information-system']
   role_ablation: 5 conditions × 5 datasets × 3 rounds = 75 runs
   interaction_order: 4 conditions × 5 datasets × 3 rounds = 60 runs
   role_specialization: 3 conditions × 5 datasets × 3 rounds = 45 runs
   shared_memory: 4 conditions × 5 datasets × 3 rounds = 60 runs
   role_diversity: 4 conditions × 5 datasets × 3 rounds = 60 runs

📊 Total Expected Runs: 300
📁 Results directory: c:\Users\kamil\Documents\Kamil\HICSS extension\repo\C4-Modeler-LLM-Agents\data\results\complete

## 🧪 Step 2: Individual Experiment Execution

Execute each experiment systematically with detailed progress tracking and error handling.

In [11]:
# ========================================================================
# STEP 2: INDIVIDUAL EXPERIMENT EXECUTION WITH PROGRESS TRACKING
# ========================================================================

def run_complete_experimental_study(model_name=EXPERIMENT_MODEL, save_intermediate=True):
    """
    Execute the complete experimental study with detailed progress tracking.
    
    Args:
        model_name: LLM model to use for all experiments
        save_intermediate: Save results after each experiment completion
    """
    
    study_start_time = datetime.now()
    study_session_id = f"complete_study_{study_start_time.strftime('%Y%m%d_%H%M%S')}"
    
    print("🚀 STARTING COMPLETE EXPERIMENTAL STUDY")
    print("=" * 80)
    print(f"📅 Study ID: {study_session_id}")
    print(f"🤖 Model: {model_name}")
    print(f"⏰ Start Time: {study_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Initialize study tracking
    study_results = {
        "study_metadata": {
            "session_id": study_session_id,
            "model_name": model_name,
            "start_time": study_start_time.isoformat(),
            "total_planned_experiments": 5,
            "total_planned_runs": 300
        },
        "experiment_results": {},
        "study_summary": {},
        "execution_log": []
    }
    
    # Define experiment execution order (strategic ordering)
    experiment_sequence = [
        ("role_ablation", "Role Ablation Study", 75),
        ("interaction_order", "Interaction Order Study", 60), 
        ("specialization", "Agent Specialization Study", 60),
        ("shared_memory", "Shared Memory Study", 45),
        ("diversity", "Team Diversity Study", 60)
    ]
    
    total_completed_runs = 0
    
    print(f"\n📋 Experiment Sequence ({len(experiment_sequence)} experiments):")
    for i, (exp_id, exp_name, exp_runs) in enumerate(experiment_sequence, 1):
        print(f"   {i}. {exp_name} ({exp_id}) - {exp_runs} runs")
    
    print("\n" + "=" * 80)
    
    # Execute each experiment in sequence
    for exp_index, (experiment_id, experiment_name, expected_runs) in enumerate(experiment_sequence, 1):
        
        exp_start_time = datetime.now()
        print(f"\n🧪 EXPERIMENT {exp_index}/5: {experiment_name}")
        print(f"📊 Expected Runs: {expected_runs}")
        print(f"⏰ Start: {exp_start_time.strftime('%H:%M:%S')}")
        print("-" * 60)
        
        # Set up experiment-specific tracing
        with setup_experiment_tracing(f"study-{experiment_id}", model_name):
            try:
                # Execute experiment
                exp_results = runner.run_experiment(experiment_id, model_name)
                
                # Calculate execution time
                exp_end_time = datetime.now()
                exp_duration = exp_end_time - exp_start_time
                
                # Extract actual runs completed
                actual_runs = 0
                if "conditions" in exp_results:
                    for condition_data in exp_results["conditions"].values():
                        actual_runs += condition_data.get("condition_summary", {}).get("total_runs", 0)
                
                total_completed_runs += actual_runs
                
                # Log success
                execution_entry = {
                    "experiment_id": experiment_id,
                    "experiment_name": experiment_name,
                    "status": "completed",
                    "start_time": exp_start_time.isoformat(),
                    "end_time": exp_end_time.isoformat(),
                    "duration_minutes": exp_duration.total_seconds() / 60,
                    "expected_runs": expected_runs,
                    "actual_runs": actual_runs,
                    "success_rate": (actual_runs / expected_runs) * 100 if expected_runs > 0 else 0
                }
                
                study_results["experiment_results"][experiment_id] = exp_results
                study_results["execution_log"].append(execution_entry)
                
                print(f"✅ COMPLETED: {experiment_name}")
                print(f"   ⏱️ Duration: {exp_duration.total_seconds()/60:.1f} minutes")
                print(f"   📊 Runs: {actual_runs}/{expected_runs} ({execution_entry['success_rate']:.1f}%)")
                print(f"   🎯 Progress: {total_completed_runs}/300 total runs ({(total_completed_runs/300)*100:.1f}%)")
                
                # Save intermediate results
                if save_intermediate:
                    intermediate_file = results_dir / f"{study_session_id}_progress_{exp_index}.json"
                    with open(intermediate_file, 'w', encoding='utf-8') as f:
                        json.dump(study_results, f, indent=2, ensure_ascii=False)
                    print(f"   💾 Progress saved: {intermediate_file.name}")
                
                # Wait for traces to complete
                wait_for_traces()
                
            except Exception as e:
                exp_end_time = datetime.now()
                exp_duration = exp_end_time - exp_start_time
                
                # Log failure
                execution_entry = {
                    "experiment_id": experiment_id,
                    "experiment_name": experiment_name,
                    "status": "failed",
                    "start_time": exp_start_time.isoformat(),
                    "end_time": exp_end_time.isoformat(),
                    "duration_minutes": exp_duration.total_seconds() / 60,
                    "expected_runs": expected_runs,
                    "actual_runs": 0,
                    "error": str(e)
                }
                
                study_results["execution_log"].append(execution_entry)
                
                print(f"❌ FAILED: {experiment_name}")
                print(f"   🚨 Error: {str(e)}")
                print(f"   ⏱️ Duration: {exp_duration.total_seconds()/60:.1f} minutes")
                
                # Continue with next experiment
                continue
        
        # Brief pause between experiments
        if exp_index < len(experiment_sequence):
            print(f"⏸️ Brief pause before next experiment...")
            time.sleep(30)  # 30-second pause
    
    # Finalize study
    study_end_time = datetime.now()
    total_duration = study_end_time - study_start_time
    
    study_results["study_metadata"]["end_time"] = study_end_time.isoformat()
    study_results["study_metadata"]["total_duration_hours"] = total_duration.total_seconds() / 3600
    study_results["study_metadata"]["completed_experiments"] = len([log for log in study_results["execution_log"] if log["status"] == "completed"])
    study_results["study_metadata"]["total_completed_runs"] = total_completed_runs
    study_results["study_metadata"]["overall_success_rate"] = (total_completed_runs / 300) * 100
    
    # Generate study summary
    study_results["study_summary"] = generate_study_summary(study_results)
    
    # Save final results
    final_results_file = results_dir / f"{study_session_id}_final_results.json"
    with open(final_results_file, 'w', encoding='utf-8') as f:
        json.dump(study_results, f, indent=2, ensure_ascii=False)
    
    print("\n" + "=" * 80)
    print("🎉 EXPERIMENTAL STUDY COMPLETED!")
    print("=" * 80)
    print(f"⏰ Total Duration: {total_duration.total_seconds()/3600:.2f} hours")
    print(f"📊 Completed Runs: {total_completed_runs}/300 ({(total_completed_runs/300)*100:.1f}%)")
    print(f"✅ Successful Experiments: {study_results['study_metadata']['completed_experiments']}/5")
    print(f"💾 Final Results: {final_results_file}")
    print("=" * 80)
    
    return study_results

def generate_study_summary(study_results):
    """Generate comprehensive study summary statistics."""
    
    summary = {
        "execution_summary": {},
        "performance_metrics": {},
        "recommendations": []
    }
    
    # Execution summary
    total_experiments = len(study_results["execution_log"])
    completed_experiments = len([log for log in study_results["execution_log"] if log["status"] == "completed"])
    failed_experiments = total_experiments - completed_experiments
    
    summary["execution_summary"] = {
        "total_experiments": total_experiments,
        "completed_experiments": completed_experiments,
        "failed_experiments": failed_experiments,
        "completion_rate": (completed_experiments / total_experiments) * 100 if total_experiments > 0 else 0
    }
    
    # Performance metrics
    if completed_experiments > 0:
        durations = [log["duration_minutes"] for log in study_results["execution_log"] if log["status"] == "completed"]
        summary["performance_metrics"] = {
            "average_experiment_duration_minutes": sum(durations) / len(durations),
            "fastest_experiment_minutes": min(durations),
            "slowest_experiment_minutes": max(durations),
            "total_execution_time_hours": sum(durations) / 60
        }
    
    return summary

# Ready to execute - uncomment the line below to start the complete study
print("💡 To start the complete experimental study, uncomment and run:")
print("# study_results = run_complete_experimental_study()")
print("\n⚠️ This will take approximately 6-8 hours to complete all 300 runs")
print("💰 Estimated cost: ~$50-80 with Google API credits")

💡 To start the complete experimental study, uncomment and run:
# study_results = run_complete_experimental_study()

⚠️ This will take approximately 6-8 hours to complete all 300 runs
💰 Estimated cost: ~$50-80 with Google API credits


## 🔄 Step 3: Alternative Execution Approaches

Choose your preferred execution method based on your time constraints and monitoring preferences.

In [12]:
# ========================================================================
# STEP 3: ALTERNATIVE EXECUTION APPROACHES
# ========================================================================

# Option A: Sequential Single Experiments (for careful monitoring)
def run_experiments_one_by_one():
    """Execute experiments one by one for careful monitoring and debugging."""
    
    experiments = ["role_ablation", "interaction_order", "specialization", "shared_memory", "diversity"]
    results = {}
    
    print("🔍 SEQUENTIAL EXECUTION MODE")
    print("Execute one experiment at a time for careful monitoring")
    print("-" * 60)
    
    for i, exp_name in enumerate(experiments, 1):
        print(f"\n▶️ Ready to run Experiment {i}/5: {exp_name}")
        print(f"   Uncomment the line below and run to execute:")
        print(f"   # results['{exp_name}'] = run_single_experiment_with_tracing('{exp_name}', 'gemini-2.5-flash')")
        print(f"   Expected duration: ~45-90 minutes")
        print(f"   Expected runs: varies by experiment")
    
    return "Use this approach for step-by-step execution with full control"

# Option B: Batch Execution (faster but less control)  
def run_experiments_in_batches():
    """Execute experiments in batches for efficiency."""
    
    print("⚡ BATCH EXECUTION MODE")
    print("Execute multiple experiments efficiently")
    print("-" * 60)
    
    # Small batch (2-3 experiments)
    print("\n🔄 Small Batch Approach:")
    print("# batch_1 = ['role_ablation', 'interaction_order']")
    print("# batch_2 = ['specialization', 'shared_memory']") 
    print("# batch_3 = ['diversity']")
    print("# results_1 = run_batch_experiments(batch_1, 'gemini-2.5-flash')")
    print("# results_2 = run_batch_experiments(batch_2, 'gemini-2.5-flash')")
    print("# results_3 = run_batch_experiments(batch_3, 'gemini-2.5-flash')")
    
    # All at once (if you're feeling brave)
    print("\n🚀 All-at-Once Approach:")
    print("# all_experiments = ['role_ablation', 'interaction_order', 'specialization', 'shared_memory', 'diversity']")
    print("# all_results = run_batch_experiments(all_experiments, 'gemini-2.5-flash')")
    
    return "Use this for faster execution with automatic error handling"

# Option C: Resume from Checkpoint
def resume_from_checkpoint():
    """Resume execution from a previous checkpoint."""
    
    print("🔄 CHECKPOINT RESUME MODE")
    print("Continue from where you left off")
    print("-" * 60)
    
    print("1. Check for existing progress files:")
    print("   progress_files = list(Path('data/results/complete_study').glob('*progress*.json'))")
    
    print("\n2. Load the latest checkpoint:")
    print("   # with open(latest_progress_file, 'r') as f:")
    print("   #     checkpoint_data = json.load(f)")
    
    print("\n3. Resume execution:")
    print("   # completed_experiments = list(checkpoint_data['experiment_results'].keys())")
    print("   # remaining_experiments = [exp for exp in all_experiments if exp not in completed_experiments]")
    print("   # remaining_results = run_batch_experiments(remaining_experiments, 'gemini-2.5-flash')")
    
    return "Use this to continue interrupted studies"

# Display execution options
print("🎯 EXECUTION OPTIONS SUMMARY")
print("=" * 80)

print("\n📋 Choose your execution approach:")
print("1. 🔄 Complete Study (Recommended):")
print("   study_results = run_complete_experimental_study()")

print("\n2. 🔍 Sequential (Step-by-Step):")
print("   run_experiments_one_by_one()")

print("\n3. ⚡ Batch Execution:")
print("   run_experiments_in_batches()")

print("\n4. 🔄 Resume from Checkpoint:")
print("   resume_from_checkpoint()")

print("\n💡 RECOMMENDATION:")
print("For your first complete study, use Option 1 (Complete Study)")
print("It provides the best balance of automation and monitoring.")

🎯 EXECUTION OPTIONS SUMMARY

📋 Choose your execution approach:
1. 🔄 Complete Study (Recommended):
   study_results = run_complete_experimental_study()

2. 🔍 Sequential (Step-by-Step):
   run_experiments_one_by_one()

3. ⚡ Batch Execution:
   run_experiments_in_batches()

4. 🔄 Resume from Checkpoint:
   resume_from_checkpoint()

💡 RECOMMENDATION:
For your first complete study, use Option 1 (Complete Study)
It provides the best balance of automation and monitoring.


## 📊 Step 4: Real-Time Monitoring and Progress Tracking

Monitor your experimental study progress with live updates and status checks.

In [13]:
# ========================================================================
# STEP 4: REAL-TIME MONITORING AND PROGRESS TRACKING
# ========================================================================

import glob
from IPython.display import clear_output
import matplotlib.pyplot as plt

def check_study_progress():
    """Check the current progress of the experimental study."""
    
    progress_files = list(Path("data/results/complete_study").glob("*progress*.json"))
    
    if not progress_files:
        print("📋 No progress files found. Study not started yet.")
        return
    
    # Get latest progress file
    latest_file = max(progress_files, key=lambda p: p.stat().st_mtime)
    
    try:
        with open(latest_file, 'r', encoding='utf-8') as f:
            progress_data = json.load(f)
        
        print("📊 CURRENT STUDY PROGRESS")
        print("=" * 60)
        
        # Study metadata
        metadata = progress_data.get("study_metadata", {})
        print(f"🆔 Study ID: {metadata.get('session_id', 'Unknown')}")
        print(f"🤖 Model: {metadata.get('model_name', 'Unknown')}")
        print(f"📅 Started: {metadata.get('start_time', 'Unknown')}")
        
        # Execution progress
        execution_log = progress_data.get("execution_log", [])
        completed = len([log for log in execution_log if log["status"] == "completed"])
        failed = len([log for log in execution_log if log["status"] == "failed"])
        total_planned = metadata.get("total_planned_experiments", 5)
        
        print(f"\n🎯 Experiment Progress: {completed + failed}/{total_planned}")
        print(f"   ✅ Completed: {completed}")
        print(f"   ❌ Failed: {failed}")
        print(f"   ⏳ Remaining: {total_planned - completed - failed}")
        
        # Runs progress
        total_runs = sum([log.get("actual_runs", 0) for log in execution_log])
        total_planned_runs = metadata.get("total_planned_runs", 300)
        
        print(f"\n📈 Runs Progress: {total_runs}/{total_planned_runs} ({(total_runs/total_planned_runs)*100:.1f}%)")
        
        # Recent activity
        if execution_log:
            latest_exp = execution_log[-1]
            print(f"\n🔄 Latest Activity:")
            print(f"   Experiment: {latest_exp.get('experiment_name', 'Unknown')}")
            print(f"   Status: {latest_exp.get('status', 'Unknown')}")
            print(f"   Duration: {latest_exp.get('duration_minutes', 0):.1f} minutes")
            if latest_exp.get('status') == 'completed':
                print(f"   Runs: {latest_exp.get('actual_runs', 0)}/{latest_exp.get('expected_runs', 0)}")
        
        print(f"\n📁 Progress file: {latest_file.name}")
        
    except Exception as e:
        print(f"❌ Error reading progress file: {str(e)}")

def visualize_study_progress():
    """Create visual progress charts for the study."""
    
    progress_files = list(Path("data/results/complete_study").glob("*progress*.json"))
    
    if not progress_files:
        print("📋 No progress data available for visualization.")
        return
    
    # Load latest progress
    latest_file = max(progress_files, key=lambda p: p.stat().st_mtime)
    
    try:
        with open(latest_file, 'r', encoding='utf-8') as f:
            progress_data = json.load(f)
        
        execution_log = progress_data.get("execution_log", [])
        
        if not execution_log:
            print("📋 No execution data available for visualization.")
            return
        
        # Create progress visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Chart 1: Experiment completion status
        experiment_names = [log["experiment_name"] for log in execution_log]
        statuses = [log["status"] for log in execution_log]
        colors = ['green' if status == 'completed' else 'red' for status in statuses]
        
        ax1.barh(experiment_names, [1] * len(experiment_names), color=colors, alpha=0.7)
        ax1.set_title('Experiment Completion Status')
        ax1.set_xlabel('Status')
        
        # Chart 2: Runs completion by experiment
        experiment_names = [log["experiment_name"] for log in execution_log if log["status"] == "completed"]
        actual_runs = [log.get("actual_runs", 0) for log in execution_log if log["status"] == "completed"]
        expected_runs = [log.get("expected_runs", 0) for log in execution_log if log["status"] == "completed"]
        
        if experiment_names:
            x = range(len(experiment_names))
            ax2.bar([i - 0.2 for i in x], expected_runs, 0.4, label='Expected', alpha=0.7, color='lightblue')
            ax2.bar([i + 0.2 for i in x], actual_runs, 0.4, label='Actual', alpha=0.7, color='darkblue')
            ax2.set_title('Runs Completion by Experiment')
            ax2.set_xlabel('Experiments')
            ax2.set_ylabel('Number of Runs')
            ax2.set_xticks(x)
            ax2.set_xticklabels([name.replace(' Study', '') for name in experiment_names], rotation=45)
            ax2.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        total_completed_runs = sum(actual_runs) if actual_runs else 0
        total_expected_runs = 300
        
        print(f"\n📊 Progress Summary:")
        print(f"   Completed Experiments: {len([log for log in execution_log if log['status'] == 'completed'])}/5")
        print(f"   Total Runs Completed: {total_completed_runs}/300 ({(total_completed_runs/300)*100:.1f}%)")
        
        if execution_log:
            avg_duration = sum([log.get("duration_minutes", 0) for log in execution_log if log["status"] == "completed"])
            completed_count = len([log for log in execution_log if log["status"] == "completed"])
            if completed_count > 0:
                avg_duration /= completed_count
                remaining_experiments = 5 - len(execution_log)
                estimated_time_remaining = remaining_experiments * avg_duration
                print(f"   Average Experiment Duration: {avg_duration:.1f} minutes")
                print(f"   Estimated Time Remaining: {estimated_time_remaining:.1f} minutes ({estimated_time_remaining/60:.1f} hours)")
        
    except Exception as e:
        print(f"❌ Error creating visualization: {str(e)}")

def monitor_study_live(refresh_interval=60):
    """Live monitoring with automatic refresh (for long-running studies)."""
    
    print("🔄 LIVE STUDY MONITORING")
    print(f"Refreshing every {refresh_interval} seconds. Press Ctrl+C to stop.")
    print("-" * 60)
    
    try:
        while True:
            clear_output(wait=True)
            print(f"🕐 Last Update: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            check_study_progress()
            print(f"\n⏰ Next refresh in {refresh_interval} seconds...")
            time.sleep(refresh_interval)
    except KeyboardInterrupt:
        print("\n🛑 Live monitoring stopped.")

# Ready to use monitoring functions
print("📊 MONITORING FUNCTIONS AVAILABLE:")
print("=" * 60)
print("1. 📋 Check current progress:")
print("   check_study_progress()")
print("\n2. 📈 Visualize progress:")
print("   visualize_study_progress()")
print("\n3. 🔄 Live monitoring:")
print("   monitor_study_live(refresh_interval=120)  # Updates every 2 minutes")
print("\n💡 Run these functions while your study is executing to track progress!")

📊 MONITORING FUNCTIONS AVAILABLE:
1. 📋 Check current progress:
   check_study_progress()

2. 📈 Visualize progress:
   visualize_study_progress()

3. 🔄 Live monitoring:
   monitor_study_live(refresh_interval=120)  # Updates every 2 minutes

💡 Run these functions while your study is executing to track progress!


## 🎯 Step 5: Execute Complete Study

**Ready to start your 300-run experimental study!**

Choose one of the execution methods below and monitor progress using the tracking functions above.

In [14]:
# ========================================================================
# STEP 5: EXECUTE COMPLETE STUDY
# ========================================================================

print("🚀 READY TO START COMPLETE EXPERIMENTAL STUDY")
print("=" * 80)
print("📊 Configuration Summary:")
print(f"   Model: {EXPERIMENT_MODEL}")
print(f"   Total Experiments: 5")
print(f"   Total Runs: 300")
print(f"   Estimated Duration: 6-8 hours")
print(f"   Estimated Cost: $50-80 (with Gemini-2.5-Flash)")
print("=" * 80)

print("\n🎯 EXECUTION COMMAND:")
print("Uncomment and run the line below to start the complete study:")
print()
print("# study_results = run_complete_experimental_study()")
print()

print("⚠️ IMPORTANT NOTES:")
print("• This will run for 6-8 hours continuously")
print("• Progress will be saved automatically after each experiment")
print("• You can monitor progress using the functions above")
print("• Results will be saved to data/results/complete_study/")
print("• LangSmith tracing will capture all interactions")
print("• Using Google's latest Gemini-2.5-Flash model for optimal performance")

print("\n📋 MONITORING DURING EXECUTION:")
print("Run these in separate cells while the study is running:")
print("• check_study_progress()  # Check current status")
print("• visualize_study_progress()  # See progress charts")
print("• monitor_study_live(120)  # Live updates every 2 minutes")

print("\n🔄 ALTERNATIVE APPROACHES:")
print("If you prefer step-by-step execution:")
print("• Use the Sequential approach from Step 3")
print("• Run experiments individually for more control")

print("\n" + "=" * 80)
print("🎯 WHEN READY, UNCOMMENT AND EXECUTE:")
print("study_results = run_complete_experimental_study()")
print("=" * 80)

🚀 READY TO START COMPLETE EXPERIMENTAL STUDY
📊 Configuration Summary:
   Model: gemini-2.5-flash-lite
   Total Experiments: 5
   Total Runs: 300
   Estimated Duration: 6-8 hours
   Estimated Cost: $50-80 (with Gemini-2.5-Flash)

🎯 EXECUTION COMMAND:
Uncomment and run the line below to start the complete study:

# study_results = run_complete_experimental_study()

⚠️ IMPORTANT NOTES:
• This will run for 6-8 hours continuously
• Progress will be saved automatically after each experiment
• You can monitor progress using the functions above
• Results will be saved to data/results/complete_study/
• LangSmith tracing will capture all interactions
• Using Google's latest Gemini-2.5-Flash model for optimal performance

📋 MONITORING DURING EXECUTION:
Run these in separate cells while the study is running:
• check_study_progress()  # Check current status
• visualize_study_progress()  # See progress charts
• monitor_study_live(120)  # Live updates every 2 minutes

🔄 ALTERNATIVE APPROACHES:
If you

## 📈 Step 6: Post-Study Analysis

After your experimental study completes, use these functions to analyze your comprehensive results.

In [15]:
# ========================================================================
# STEP 6: POST-STUDY ANALYSIS
# ========================================================================

def load_complete_study_results():
    """Load and analyze results from the complete experimental study."""
    
    study_dir = Path("data/results/complete_study")
    final_results_files = list(study_dir.glob("*final_results.json"))
    
    if not final_results_files:
        print("❌ No final results found. Study may not be complete yet.")
        return None
    
    # Load the most recent study
    latest_study = max(final_results_files, key=lambda p: p.stat().st_mtime)
    
    try:
        with open(latest_study, 'r', encoding='utf-8') as f:
            study_data = json.load(f)
        
        print("📊 COMPLETE STUDY RESULTS LOADED")
        print("=" * 60)
        
        metadata = study_data.get("study_metadata", {})
        print(f"🆔 Study ID: {metadata.get('session_id', 'Unknown')}")
        print(f"🤖 Model: {metadata.get('model_name', 'Unknown')}")
        print(f"📅 Duration: {metadata.get('total_duration_hours', 0):.2f} hours")
        print(f"📊 Success Rate: {metadata.get('overall_success_rate', 0):.1f}%")
        print(f"🎯 Completed Runs: {metadata.get('total_completed_runs', 0)}/300")
        
        print(f"\n📁 Results file: {latest_study}")
        
        return study_data
        
    except Exception as e:
        print(f"❌ Error loading study results: {str(e)}")
        return None

def analyze_complete_study_results(study_data):
    """Perform comprehensive analysis of the complete study results."""
    
    if not study_data:
        print("❌ No study data provided")
        return
    
    print("🔬 COMPREHENSIVE STUDY ANALYSIS")
    print("=" * 60)
    
    # Extract all experimental results into unified format
    all_experiment_data = {}
    for exp_id, exp_results in study_data.get("experiment_results", {}).items():
        all_experiment_data[exp_id] = exp_results
    
    # Use existing analysis functions from Section 6
    print("📊 Extracting metrics from all experiments...")
    unified_metrics_df = extract_metrics_dataframe(all_experiment_data)
    
    if unified_metrics_df.empty:
        print("❌ No metrics data available for analysis")
        return unified_metrics_df
    
    print(f"✅ Extracted metrics from {len(unified_metrics_df)} experimental conditions")
    
    # Display overview
    print(f"\n📋 METRICS OVERVIEW:")
    print(f"   Total Conditions: {len(unified_metrics_df)}")
    print(f"   Experiments: {unified_metrics_df['experiment'].nunique()}")
    print(f"   Total Runs Analyzed: {unified_metrics_df['total_runs'].sum()}")
    
    # Show experiment breakdown
    exp_summary = unified_metrics_df.groupby('experiment').agg({
        'total_runs': 'sum',
        'condition': 'count'
    }).rename(columns={'condition': 'conditions'})
    
    print(f"\n📈 EXPERIMENT BREAKDOWN:")
    for exp_name, data in exp_summary.iterrows():
        print(f"   {exp_name}: {data['conditions']} conditions, {data['total_runs']} runs")
    
    return unified_metrics_df

def create_comprehensive_study_report(study_data, metrics_df):
    """Generate a comprehensive publication-ready report."""
    
    if study_data is None or metrics_df is None or metrics_df.empty:
        print("❌ Insufficient data for comprehensive report")
        return
    
    print("📑 GENERATING COMPREHENSIVE STUDY REPORT")
    print("=" * 60)
    
    # Create multiple visualizations
    fig, axes = plt.subplots(3, 2, figsize=(20, 18))
    fig.suptitle('Complete Experimental Study: Journal Extension Results', fontsize=16, fontweight='bold')
    
    # 1. Overall performance by experiment
    ax1 = axes[0, 0]
    exp_performance = metrics_df.groupby('experiment')[['iso_functional', 'iso_maintainability', 'abstraction_coherence']].mean()
    exp_performance.plot(kind='bar', ax=ax1)
    ax1.set_title('Average Quality Metrics by Experiment')
    ax1.set_ylabel('Score')
    ax1.tick_params(axis='x', rotation=45)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 2. ISO 25010 heatmap across all experiments
    ax2 = axes[0, 1]
    iso_metrics = [col for col in metrics_df.columns if col.startswith('iso_')]
    if iso_metrics:
        iso_summary = metrics_df.groupby('experiment')[iso_metrics].mean()
        sns.heatmap(iso_summary.T, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax2)
        ax2.set_title('ISO 25010 Metrics Heatmap (All Experiments)')
    
    # 3. Architectural soundness comparison
    ax3 = axes[1, 0]
    soundness_metrics = ['coupling_density', 'abstraction_coherence', 'semantic_completeness']
    soundness_summary = metrics_df.groupby('experiment')[soundness_metrics].mean()
    soundness_summary.plot(kind='bar', ax=ax3)
    ax3.set_title('Architectural Soundness Metrics')
    ax3.set_ylabel('Score')
    ax3.tick_params(axis='x', rotation=45)
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 4. Condition comparison within experiments
    ax4 = axes[1, 1]
    if 'role_ablation' in metrics_df['experiment'].values:
        role_data = metrics_df[metrics_df['experiment'].str.contains('role_ablation', case=False, na=False)]
        if not role_data.empty:
            role_data.plot(x='condition', y='iso_functional', kind='bar', ax=ax4, color='skyblue')
            ax4.set_title('Role Ablation: Functional Suitability')
            ax4.set_ylabel('ISO Functional Suitability')
            ax4.tick_params(axis='x', rotation=45)
    
    # 5. ATAM responsiveness across experiments
    ax5 = axes[2, 0]
    atam_summary = metrics_df.groupby('experiment')['atam_responsiveness'].mean()
    atam_summary.plot(kind='bar', ax=ax5, color='lightgreen')
    ax5.set_title('ATAM Scenario Responsiveness by Experiment')
    ax5.set_ylabel('Responsiveness Score')
    ax5.tick_params(axis='x', rotation=45)
    
    # 6. Overall composite quality score
    ax6 = axes[2, 1]
    # Calculate composite quality score
    quality_metrics = ['iso_functional', 'iso_maintainability', 'abstraction_coherence', 'atam_responsiveness']
    available_quality_metrics = [col for col in quality_metrics if col in metrics_df.columns]
    
    if available_quality_metrics:
        metrics_df['composite_quality'] = metrics_df[available_quality_metrics].mean(axis=1)
        composite_summary = metrics_df.groupby('experiment')['composite_quality'].mean()
        composite_summary.plot(kind='bar', ax=ax6, color='orange')
        ax6.set_title('Composite Quality Score by Experiment')
        ax6.set_ylabel('Composite Score')
        ax6.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Generate statistical summary
    print("\n📊 STATISTICAL SUMMARY:")
    summary_stats = metrics_df[available_quality_metrics + ['iso_security']].describe()
    display(summary_stats)
    
    # Generate ANOVA analysis for key metrics
    print("\n📈 STATISTICAL SIGNIFICANCE ANALYSIS:")
    for metric in ['iso_functional', 'abstraction_coherence', 'atam_responsiveness']:
        if metric in metrics_df.columns:
            print(f"\n{metric.upper()}:")
            try:
                # Group data by experiment
                groups = [group[metric].values for name, group in metrics_df.groupby('experiment')]
                if len(groups) >= 2:
                    f_stat, p_value = stats.f_oneway(*groups)
                    print(f"   F-statistic: {f_stat:.4f}")
                    print(f"   P-value: {p_value:.4f}")
                    print(f"   Significant: {'Yes' if p_value < 0.05 else 'No'} (α = 0.05)")
                else:
                    print("   Insufficient data for ANOVA")
            except Exception as e:
                print(f"   Error in ANOVA: {str(e)}")
    
    # Export comprehensive results
    export_file = f"data/results/complete_study_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    metrics_df.to_csv(export_file, index=False)
    print(f"\n💾 Comprehensive results exported to: {export_file}")
    
    return metrics_df

# Ready to analyze results
print("📈 POST-STUDY ANALYSIS FUNCTIONS:")
print("=" * 60)
print("1. 📊 Load study results:")
print("   study_data = load_complete_study_results()")
print("\n2. 🔬 Analyze results:")
print("   metrics_df = analyze_complete_study_results(study_data)")
print("\n3. 📑 Generate comprehensive report:")
print("   final_analysis = create_comprehensive_study_report(study_data, metrics_df)")
print("\n💡 Run these functions after your experimental study completes!")

📈 POST-STUDY ANALYSIS FUNCTIONS:
1. 📊 Load study results:
   study_data = load_complete_study_results()

2. 🔬 Analyze results:
   metrics_df = analyze_complete_study_results(study_data)

3. 📑 Generate comprehensive report:
   final_analysis = create_comprehensive_study_report(study_data, metrics_df)

💡 Run these functions after your experimental study completes!


In [17]:
# ========================================================================
# EXECUTE COMPLETE EXPERIMENTAL STUDY - ALL 300 RUNS WITH GEMINI-2.5-FLASH-LITE
# ========================================================================

print("🚀 STARTING COMPLETE EXPERIMENTAL STUDY")
print("=" * 80)
print("📊 About to execute 300 experimental runs for journal research")
print("🤖 Model: gemini-2.5-flash-lite")
print("⏰ Estimated duration: 6-8 hours")
print("💰 Estimated cost: ~$50-80 (excellent performance with Google's latest model)")
print("=" * 80)

# Execute the complete study with Gemini-2.5-Flash-Lite
study_results = run_complete_experimental_study()

🚀 STARTING COMPLETE EXPERIMENTAL STUDY
📊 About to execute 300 experimental runs for journal research
🤖 Model: gemini-2.5-flash-lite
⏰ Estimated duration: 6-8 hours
💰 Estimated cost: ~$50-80 (excellent performance with Google's latest model)
🚀 STARTING COMPLETE EXPERIMENTAL STUDY
📅 Study ID: complete_study_20251022_140809
🤖 Model: gemini-2.5-flash-lite
⏰ Start Time: 2025-10-22 14:08:09

📋 Experiment Sequence (5 experiments):
   1. Role Ablation Study (role_ablation) - 75 runs
   2. Interaction Order Study (interaction_order) - 60 runs
   3. Agent Specialization Study (specialization) - 60 runs
   4. Shared Memory Study (shared_memory) - 45 runs
   5. Team Diversity Study (diversity) - 60 runs


🧪 EXPERIMENT 1/5: Role Ablation Study
📊 Expected Runs: 75
⏰ Start: 14:08:09
------------------------------------------------------------

🧪 Starting Enhanced Experiment: Role Ablation Study
📋 Description: Remove one expert role and evaluate impact on C4 model quality
🎯 Purpose: Determine which ro

KeyboardInterrupt: 